# Natya Posture Alignment - Static Postures Training

Run this notebook in Google Colab to train the model on STATIC POSTURES (images).


In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

!pip install mediapipe opencv-python-headless pandas scikit-learn seaborn matplotlib tqdm


In [ ]:
# 2. Imports and Configuration
import os, glob, pickle, warnings, re
import numpy as np
import pandas as pd
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from collections import defaultdict
warnings.filterwarnings('ignore')

# ----------------- CONFIGURATION -----------------
DRIVE_ROOT = '/content/drive/MyDrive/TrainingData'
IMAGES_DIR = f'{DRIVE_ROOT}/FinalPostures'
CSV_PATH = f'{DRIVE_ROOT}/Instructions/static_postures_template.csv'
CHECKPOINT_DIR = f'{DRIVE_ROOT}/Checkpoints'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MIN_VISIBILITY = 0.50
MAX_BAD_JOINT_FRAC = 0.20
MIN_IMAGES = 2
MAX_PER_CLASS = None

FEATURES_CACHE = f'{CHECKPOINT_DIR}/postures_features.npz'
RAW_CACHE = f'{CHECKPOINT_DIR}/postures_raw_samples.pkl'
CKPT_PATH = f'{CHECKPOINT_DIR}/posture_model.pt'

print(f'Device: {DEVICE}')
print(f'Reading images from: {IMAGES_DIR}')
print(f'Reading CSV from: {CSV_PATH}')


In [ ]:
# 3. Angle Definitions and MediaPipe Utilities
ANGLE_DEFS = [
    ('left_shoulder',  13, 11, 23), ('right_shoulder', 14, 12, 24),
    ('left_elbow',     11, 13, 15), ('right_elbow',    12, 14, 16),
    ('left_wrist',     13, 15, 19), ('right_wrist',    14, 16, 20),
    ('left_hip',       11, 23, 25), ('right_hip',      12, 24, 26),
    ('left_knee',      23, 25, 27), ('right_knee',     24, 26, 28),
    ('left_ankle',     25, 27, 31), ('right_ankle',    26, 28, 32),
]
ANGLE_NAMES = [d[0] for d in ANGLE_DEFS]

# Feature Dim for STATIC Image: 33 joints*2 coords = 66 + 12 angles + 6 sym = 84
FEATURE_DIM = 66 + 12 + 6

# MediaPipe Initialization
!wget -q -O pose_landmarker_heavy.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task

base_options = python.BaseOptions(model_asset_path='pose_landmarker_heavy.task')
options = vision.PoseLandmarkerOptions(base_options=base_options, output_segmentation_masks=False, num_poses=1)
mp_pose = vision.PoseLandmarker.create_from_options(options)
print('MediaPipe Pose model ready.')

def _angle_between(pa, pv, pc):
    v1 = pa - pv;  v2 = pc - pv
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 < 1e-6 or n2 < 1e-6: return 0.0
    cos_a = np.clip(np.dot(v1, v2) / (n1 * n2), -1.0, 1.0)
    return float(np.degrees(np.arccos(cos_a)))

def compute_angles_for_frame(frame):
    return np.array([_angle_between(frame[a, :2], frame[v, :2], frame[c, :2]) for _, a, v, c in ANGLE_DEFS])

def normalise_landmarks(seq):
    seq = seq.copy()
    hip_mid = (seq[23, :2] + seq[24, :2]) / 2
    shoulder_mid = (seq[11, :2] + seq[12, :2]) / 2
    scale = np.linalg.norm(shoulder_mid - hip_mid)
    scale = max(scale, 1e-6)
    seq[:, :2] = (seq[:, :2] - hip_mid) / scale
    return seq

def pad_to_square(image: np.ndarray) -> np.ndarray:
    h, w = image.shape[:2]
    if h == w: return image
    size = max(h, w)
    pad_h = (size - h) // 2
    pad_w = (size - w) // 2
    return cv2.copyMakeBorder(image, pad_h, size - h - pad_h, pad_w, size - w - pad_w, cv2.BORDER_CONSTANT, value=[0, 0, 0])

def extract_landmarks_from_image(image_path):
    frame = cv2.imread(image_path)
    if frame is None: return None, None
    
    frame = pad_to_square(frame)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = mp_pose.detect(mp_image)
    
    if not result.pose_landmarks: return None, None
    
    lm = result.pose_landmarks[0]
    arr = np.array([[l.x, l.y, l.visibility] for l in lm])
    bad_frac = np.mean(arr[:, 2] < MIN_VISIBILITY)
    if bad_frac > MAX_BAD_JOINT_FRAC: return None, None
    
    seq_norm = normalise_landmarks(arr)
    angles = compute_angles_for_frame(seq_norm)
    return seq_norm, angles

SYMMETRY_PAIRS = [
    (ANGLE_NAMES.index('left_shoulder'), ANGLE_NAMES.index('right_shoulder')),
    (ANGLE_NAMES.index('left_elbow'),    ANGLE_NAMES.index('right_elbow')),
    (ANGLE_NAMES.index('left_wrist'),    ANGLE_NAMES.index('right_wrist')),
    (ANGLE_NAMES.index('left_hip'),      ANGLE_NAMES.index('right_hip')),
    (ANGLE_NAMES.index('left_knee'),     ANGLE_NAMES.index('right_knee')),
    (ANGLE_NAMES.index('left_ankle'),    ANGLE_NAMES.index('right_ankle')),
]

def compute_symmetry_features(angles):
    return np.array([abs(angles[l] - angles[r]) for l, r in SYMMETRY_PAIRS])

def build_feature_vector(seq_norm, angles):
    coords = seq_norm[:, :2].flatten()
    sym = compute_symmetry_features(angles)
    return np.concatenate([coords, angles, sym])

LR_PAIRS = [(11, 12), (13, 14), (15, 16), (17, 18), (19, 20), (21, 22), (23, 24), (25, 26), (27, 28), (29, 30), (31, 32), (1, 4), (2, 5), (3, 6), (7, 8), (9, 10)]

def flip_sequence(seq_norm):
    flipped = seq_norm.copy()
    flipped[:, 0] *= -1
    for l, r in LR_PAIRS:
        flipped[[l, r]] = flipped[[r, l]]
    return flipped

def add_noise(seq_norm, sigma=0.01):
    noisy = seq_norm.copy()
    noisy[:, :2] += np.random.normal(0, sigma, noisy[:, :2].shape)
    return noisy

def augment_sample(seq_norm, angles):
    variants = []
    flipped = flip_sequence(seq_norm)
    flip_angles = compute_angles_for_frame(flipped)
    variants.append((build_feature_vector(flipped, flip_angles), flip_angles, flipped))

    noisy = add_noise(seq_norm)
    noisy_angles = compute_angles_for_frame(noisy)
    variants.append((build_feature_vector(noisy, noisy_angles), noisy_angles, noisy))
    return variants


In [ ]:
# 4. Read Labels and Map to Images in Google Drive
print(f"Loading instructions from {CSV_PATH}")
df = pd.read_csv(CSV_PATH)
# Assuming 'Posture_Name' or similar is the target. Using the second column as label usually.
if 'Posture_Name' in df.columns:
    label_col = 'Posture_Name'
elif 'Step_Name' in df.columns:
    label_col = 'Step_Name'
else:
    label_col = df.columns[1]

id_col = df.columns[0]
df[label_col] = df[label_col].astype(str).str.strip()
df[id_col] = df[id_col].astype(str).str.strip()

print(f"Loaded {len(df)} rows. Using '{id_col}' as ID and '{label_col}' as Label.")

all_drive_images = glob.glob(f"{IMAGES_DIR}/*")
print(f"Found {len(all_drive_images)} items in directory.")

image_files = {}
for _, row in df.iterrows():
    step_id = row[id_col]
    label = row[label_col]
    
    matched_file = None
    for vf in all_drive_images:
        if step_id in os.path.basename(vf):
            matched_file = vf
            break
    if matched_file: image_files[step_id] = {'path': matched_file, 'label': label}

print(f"Successfully matched {len(image_files)} images.")

class_counts = defaultdict(int)
for v in image_files.values(): class_counts[v['label']] += 1

valid_classes = {cls for cls, cnt in class_counts.items() if cnt >= MIN_IMAGES}
processing_list = [v for v in image_files.values() if v['label'] in valid_classes]
print(f"Total images to process: {len(processing_list)}")


In [ ]:
# 5. Extract Features and Cache Data
cache_valid = False
if os.path.exists(FEATURES_CACHE):
    try:
        data = np.load(FEATURES_CACHE, allow_pickle=True)
        if data['X'].shape[1] == FEATURE_DIM:
            X = data['X']; y = data['y']
            label_names = list(data['label_names'])
            cache_valid = True
            print("Loaded from cache!")
    except Exception as e: pass

raw_samples = []
if os.path.exists(RAW_CACHE):
    try:
        with open(RAW_CACHE, 'rb') as f: raw_samples = pickle.load(f)
    except: pass

if not cache_valid:
    failed = []
    if len(raw_samples) == 0:
        print('Extracting raw image sequences...')
        for item in tqdm(processing_list):
            img_path = item['path']; cls = item['label']
            try:
                seq_norm, angles = extract_landmarks_from_image(img_path)
                if seq_norm is None:
                    failed.append(img_path); continue
                fv = build_feature_vector(seq_norm, angles)
                raw_samples.append((fv, angles, cls, seq_norm, img_path))
            except Exception as e: failed.append(img_path)

        with open(RAW_CACHE, 'wb') as f: pickle.dump(raw_samples, f)
            
    temp_y = [item[2] for item in raw_samples]
    unique_labels = sorted(set(temp_y))
    class_sample_counts = {cls: int(np.sum(np.array(temp_y) == cls)) for cls in unique_labels}
    max_count = max(class_sample_counts.values()) if class_sample_counts else 0
    print(f'Balancing classes to max_count: {max_count}')
    
    X_aug, y_aug = [], []
    class_raw = defaultdict(list)
    for item in raw_samples: class_raw[item[2]].append(item)

    for cls in unique_labels:
        items = class_raw[cls]
        for fv, angles, _, seq_norm, img_path in items:
            X_aug.append(fv); y_aug.append(cls)

        needed = max_count - len(items)
        if needed <= 0: continue

        pool = items.copy()
        added = 0
        while added < needed:
            src = pool[added % len(pool)]
            fv, angles, _, seq_norm, img_path = src
            variants = augment_sample(seq_norm, angles)
            for v_fv, v_ang, v_seq in variants:
                if added >= needed: break
                X_aug.append(v_fv); y_aug.append(cls)
                added += 1

    X = np.array(X_aug); y = np.array(y_aug)
    label_names = sorted(set(y))
    np.savez(FEATURES_CACHE, X=X, y=y, label_names=label_names)
    print(f'Saved cache → {FEATURES_CACHE}. Samples: {len(X)}')


In [ ]:
# 6. Model Definition and Training Loop
le = LabelEncoder()
y_enc = le.fit_transform(y)
NUM_CLASSES = len(le.classes_)

X_mean = X.mean(axis=0)
X_std  = X.std(axis=0) + 1e-8
X_norm = (X - X_mean) / X_std

try:
    X_train, X_val, y_train, y_val = train_test_split(X_norm, y_enc, test_size=0.2, random_state=42, stratify=y_enc)
except ValueError:
    X_train, X_val, y_train, y_val = train_test_split(X_norm, y_enc, test_size=0.2, random_state=42)

class PostureClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, hidden=128, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.BatchNorm1d(hidden // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden // 2, num_classes),
        )
    def forward(self, x): return self.net(x)

class PostureDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

model = PostureClassifier(FEATURE_DIM, NUM_CLASSES).to(DEVICE)
EPOCHS, BATCH, LR, WD = 100, 16, 1e-3, 1e-4

train_loader = DataLoader(PostureDataset(X_train, y_train), batch_size=BATCH, shuffle=True, drop_last=True if len(X_train) > BATCH else False)
val_loader   = DataLoader(PostureDataset(X_val, y_val), batch_size=BATCH, shuffle=False)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

counts_tr = np.array([np.sum(y_train == i) for i in range(NUM_CLASSES)], dtype=float)
cw = 1.0 / (counts_tr + 1e-8); cw = cw / cw.sum() * NUM_CLASSES
criterion = nn.CrossEntropyLoss(weight=torch.FloatTensor(cw).to(DEVICE))

best_val_acc, best_state = 0.0, None

for epoch in range(1, EPOCHS + 1):
    model.train()
    for Xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(Xb.to(DEVICE)), yb.to(DEVICE))
        loss.backward()
        optimizer.step()
    scheduler.step()

    model.eval()
    correct = 0
    with torch.no_grad():
        for Xb, yb in val_loader:
            correct += (model(Xb.to(DEVICE)).argmax(1) == yb.to(DEVICE)).sum().item()
    acc = correct / len(X_val) if len(X_val) > 0 else 0
    if acc > best_val_acc:
        best_val_acc = acc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    if epoch % 20 == 0: print(f'Epoch {epoch:3d}/{EPOCHS} | val_acc={acc:.2%}')

if best_state: model.load_state_dict(best_state)
print(f'Best val accuracy: {best_val_acc:.2%}')

ckpt = {
    'model_state':  best_state, 'label_encoder': le, 'X_mean': X_mean, 'X_std': X_std,
    'num_classes':  NUM_CLASSES, 'feature_dim':  FEATURE_DIM,
}
torch.save(ckpt, CKPT_PATH)
print(f'Checkpoint saved -> {CKPT_PATH}')

